In [3]:
from smartdoc15_ch1 import Dataset, evaluate_segmentation

ImportError: cannot import name '_fetch_remote' from 'sklearn.datasets' (/run/media/vlad/CE10841C10840DA1/venvs/venv/lib/python3.11/site-packages/sklearn/datasets/__init__.py)

In [1]:
d = Dataset(data_home="/data/competitions/2015-ICDAR-smartdoc/challenge1/99-computable-version-2017-test",
           download_if_missing=False)

NameError: name 'Dataset' is not defined

In [3]:
seg_gt = d.segmentation_targets
seg_gt.shape

(50, 8)

Evaluation ground truth vs ground truth

In [4]:
evaluate_segmentation(seg_gt, seg_gt, d.model_shapes, print_summary=True)

----------------------------------------------
   evaluate_segmentation: Evaluation report
----------------------------------------------
metric: IoU (aka Jaccard index)
----------------------------------------------
observations:    50
mean:             1.00 (CI@95%: 1.000, 1.000)
min-max:          1.000 - 1.000
variance:         0.000 (std: 0.000)
----------------------------------------------


array([ 1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,
        1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,
        1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,
        1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.])

Evaluation ground truth scaled vs ground truth

In [5]:
seg_reduced = d.segmentation_targets * 0.5

In [6]:
evaluate_segmentation(seg_reduced, seg_gt, d.model_shapes, frame_scale_factor=0.5, print_summary=True)

----------------------------------------------
   evaluate_segmentation: Evaluation report
----------------------------------------------
metric: IoU (aka Jaccard index)
----------------------------------------------
observations:    50
mean:             1.00 (CI@95%: 1.000, 1.000)
min-max:          1.000 - 1.000
variance:         0.000 (std: 0.000)
----------------------------------------------


array([ 1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,
        1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,
        1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,
        1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.])

The output contains the individual evaluation for each observation, making is easy to merge evaluations from different subsets, perform cross validation, identify problematic elements, or simply to extract more elaborate statistics about those values.

In [7]:
import numpy as np

In [14]:
np.tile?

In [10]:
observ = seg_gt.shape[0]
framew, frameh = 1920., 1080.

In [19]:
wholeframeseg = np.array([0., 0., 0., frameh, framew, frameh, framew, 0.])
fakeresults1 = np.tile(wholeframeseg, (observ, 1))
fakeresults1.shape

(50, 8)

In [21]:
evaluate_segmentation(fakeresults1, seg_gt, d.model_shapes, print_summary=True)

----------------------------------------------
   evaluate_segmentation: Evaluation report
----------------------------------------------
metric: IoU (aka Jaccard index)
----------------------------------------------
observations:    50
mean:             0.11 (CI@95%: 0.100, 0.119)
min-max:          0.041 - 0.173
variance:         0.001 (std: 0.034)
----------------------------------------------


array([ 0.13641666,  0.07597955,  0.07346789,  0.14002696,  0.06881893,
        0.09835207,  0.12968018,  0.08838395,  0.06561334,  0.13469754,
        0.12902699,  0.14044994,  0.09351552,  0.05851509,  0.08977727,
        0.11079546,  0.0817408 ,  0.17336802,  0.11894965,  0.13006599,
        0.13305495,  0.11265368,  0.16414012,  0.0642706 ,  0.13675877,
        0.1159726 ,  0.1670184 ,  0.11575269,  0.15434624,  0.14086942,
        0.14107757,  0.08068005,  0.08205667,  0.08849093,  0.14463279,
        0.0752603 ,  0.15033108,  0.13632017,  0.11354156,  0.0778586 ,
        0.08139904,  0.04117838,  0.14629553,  0.10258302,  0.07059613,
        0.14151006,  0.15199741,  0.07204286,  0.06628012,  0.05887742])

In [23]:
fakeresults2 = np.tile(seg_gt.mean(axis=0), (observ, 1))
fakeresults2.shape

(50, 8)

In [24]:
evaluate_segmentation(fakeresults2, seg_gt, d.model_shapes, print_summary=True)

----------------------------------------------
   evaluate_segmentation: Evaluation report
----------------------------------------------
metric: IoU (aka Jaccard index)
----------------------------------------------
observations:    50
mean:             0.46 (CI@95%: 0.423, 0.507)
min-max:          0.189 - 0.805
variance:         0.023 (std: 0.150)
----------------------------------------------


array([ 0.45183008,  0.55355323,  0.38872883,  0.36433957,  0.78977809,
        0.2646679 ,  0.36152687,  0.18923517,  0.54230772,  0.45843342,
        0.37718844,  0.43986082,  0.42138467,  0.67858408,  0.66141109,
        0.3610824 ,  0.47823445,  0.353207  ,  0.37212445,  0.34927205,
        0.46757727,  0.49855925,  0.26525091,  0.75225125,  0.43816964,
        0.52902661,  0.36518089,  0.47279823,  0.37296018,  0.44090575,
        0.44430899,  0.64669361,  0.40195558,  0.2053788 ,  0.28261313,
        0.80487323,  0.35685596,  0.24277402,  0.46944937,  0.72896937,
        0.75070575,  0.36296816,  0.4128719 ,  0.56657698,  0.55004232,
        0.43794655,  0.40503238,  0.60140644,  0.47140204,  0.64772461])

In [29]:
def seg_list_of_dict_to_array(list_of_dict):
    return np.float32([
        [e[k] 
        for k in ('tl_x', 'tl_y', 'bl_x', 'bl_y', 'br_x', 'br_y', 'tr_x', 'tr_y')]
        for e in list_of_dict])

In [30]:
seg_listdict_1 = [
    {'tl_x': 0., 'tl_y': 0., 'bl_x': 0., 'bl_y': frameh, 'br_x': framew, 'br_y':frameh, 'tr_x': framew, 'tr_y': 0.},
    {'tl_x': 0., 'tl_y': 0., 'bl_x': 0., 'bl_y': frameh, 'br_x': framew, 'br_y':frameh, 'tr_x': framew, 'tr_y': 0.},
]

In [31]:
seg_aa_1 = seg_list_of_dict_to_array(seg_listdict_1)

In [32]:
seg_aa_1

array([[    0.,     0.,     0.,  1080.,  1920.,  1080.,  1920.,     0.],
       [    0.,     0.,     0.,  1080.,  1920.,  1080.,  1920.,     0.]], dtype=float32)

In [33]:
evaluate_segmentation(seg_aa_1, seg_gt[:2], d.model_shapes[:2], print_summary=True)

----------------------------------------------
   evaluate_segmentation: Evaluation report
----------------------------------------------
metric: IoU (aka Jaccard index)
----------------------------------------------
observations:     2
mean:             0.11 (CI@95%: 0.047, 0.165)
min-max:          0.076 - 0.136
variance:         0.002 (std: 0.043)
----------------------------------------------


array([ 0.13641666,  0.07597955])